<a href="https://colab.research.google.com/github/HubbaBubba64/AI-Grammar-Score-Prototype/blob/main/week2_classification/grammar_classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers datasets torch scikit-learn pandas

In [ ]:
import pandas as pd

df = pd.read_csv("data/grammar_sample.csv")

print(df.head())

In [ ]:
label_map = {
    "bad": 0,
    "good": 1
}

df["labels"] = df["label"].map(label_map)

print(df.head())


In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

In [ ]:
encodings = tokenizer(
    list(df["text"]),
    truncation=True,
    padding=True,
    return_tensors="pt"
)

print(encodings.keys())

dict_keys(['input_ids','token_tyoe_ids','attention_mask'])

In [ ]:
import torch

labels = torch.tensor(df["labels"].values)

print(labels)

In [ ]:
from torch.utils.data import Dataset

class GrammarDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {
            key: val[idx]
            for key, val in self.encodings.items()
        }
        item["labels"] = self.labels[idx]
        return item

    def __len__(self):
        return len(self.labels)

dataset = GrammarDataset(encodings, labels)

In [ ]:
from torch.utils.data import random_split

train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size

train_dataset, test_dataset = random_split(dataset, [train_size, test_size])

print("Train size:", len(train_dataset))
print("Test size:", len(test_dataset))